In [1]:
!pip install diffusers transformers torch gradio accelerate -q

In [2]:
import torch
from diffusers import StableDiffusionPipeline
print("Libraries imported successfully!")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Libraries imported successfully!


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
model_id = "runwayml/stable-diffusion-v1-5"
pipeline = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16 if device == "cuda" else torch.float32)
pipeline = pipeline.to(device)
print("Stable Diffusion pipeline loaded successfully!")

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Stable Diffusion pipeline loaded successfully!


In [5]:
model_id = "CompVis/stable-diffusion-v1-4"

pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

print("✅ Model loaded successfully!")

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

✅ Model loaded successfully!


In [6]:
import gradio as gr
print("gradio imported successfully")

gradio imported successfully


In [7]:
import os
print("Current folder contents:")
print(os.listdir("."))

print("\nGenerated images folder:")
print(os.listdir("generated_images"))

Current folder contents:
['.config', 'sample_data']

Generated images folder:


FileNotFoundError: [Errno 2] No such file or directory: 'generated_images'

In [ ]:
import gradio as gr
from datetime import datetime
import os

os.makedirs("generated_images",exist_ok=True)
history=[]

def generate_images(prompt,negative_prompt="",steps=40,guidance=7.5,style="Realisitic"):
    if not prompt or prompt.strip()=="":
       return "Please enter a prompt",None,None,None

    #Adding style
    style_prompt = prompt

    if style=="Anime":
      style_prompt = prompt + ",Anime style"
    elif style=="Oil Painting":
      style_prompt = prompt + ",Oil Painting"
    elif style=="Cartoon":
      style_prompt = prompt + ",Cartoon"
    elif style=="Cyberpunk":
      style_prompt = prompt + ",Cyberpunk"
    elif style=="Fantasy":
      style_prompt = prompt + ",Fantasy"


    print("Generating...")
    image=pipe(
        style_prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=int(steps),
        guidance_scale=guidance
    ).images[0]

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"generated_images/ai_image_{timestamp}.png"
    image.save(filename)

    history.append(image)

    return f"Saved:{filename}({style} style)",image,image,history

interface = gr.Interface(
    fn = generate_images,
    inputs = [
        gr.Textbox(lines=4,placeholder="Describe the image you have to generate",label="prompt"),
        gr.Textbox(lines=6,placeholder="Thingd to avoid",label="Negative Prompt"),
        gr.Slider(20,60,value=40,step=5,label="Quality Step"),
        gr.Slider(1,15,value=7.5,step=0.5,label="Creativity Level"),
        gr.Dropdown(choices=["Realistic", "Anime", "Oil Painting", "Cartoon", "Cyberpunk", "Fantasy"],
                   value="Realistic", label="Style")
    ],
    outputs=[
        gr.Textbox(label="Status"),
        gr.Image(label="Generated_image"),
        gr.Image(label="Download_image"),
        gr.Gallery(label="History",height=400)
    ],
    title="AI-Image Generator-6.O",
    description="First Year AI engineeing project",
    examples=[
        ["A big brown cat jump over lazy dog","",50,7.5,"Anime"],
        ["a cute kitten meowing for milk to the owner","",40,8,"Realistic"]
    ],
    theme="soft"
)

interface.launch(share=True)


